In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/Question 2 - Leaderboard')
!ls Data

Mounted at /content/drive
optional_external_data.csv  student_test.csv  student_train.csv


In [2]:
import math, time
import numpy as np, pandas as pd, torch
from torch import nn
from torch.nn import functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PRED_LEN, SEQ_LEN, LABEL_LEN, PERIOD = 168, 168, 48, 24
VAL_BLOCKS = 52

train_df = pd.read_csv("Data/student_train.csv")
ext_df   = pd.read_csv("Data/optional_external_data.csv")
y_all = train_df["value"].to_numpy(np.float64)
N, T = len(y_all), len(ext_df)                       # 43656, 43824
VAL_START = N - VAL_BLOCKS * PRED_LEN
VAL_ORIGINS = np.array([VAL_START + i*PRED_LEN for i in range(VAL_BLOCKS)])
TRUE_VAL = np.stack([y_all[o:o+PRED_LEN] for o in VAL_ORIGINS])
print(DEVICE, N, T, "val from", VAL_START)

def build_marks(fit_end, use_external=True):
    """Per-step inputs for all T steps. Period 24 recovered from the autocorrelation."""
    e = ext_df; t = e["time_idx"].to_numpy(np.float64) - 1
    phase = np.stack([np.sin(2*np.pi*t/PERIOD), np.cos(2*np.pi*t/PERIOD),
                      np.sin(2*np.pi*t/(PERIOD*7)), np.cos(2*np.pi*t/(PERIOD*7))], 1)
    if not use_external:
        return phase.astype(np.float32)
    cont = np.stack([e.feature_A, e.feature_B, e.feature_C, np.log1p(e.feature_D),
                     np.log1p(e.feature_E), np.log1p(e.feature_F)], 1).astype(np.float64)
    onehot = e[["feature_G","feature_H","feature_I","feature_J"]].to_numpy(np.float64)
    base = pd.DataFrame(np.concatenate([cont, onehot], 1))
    blocks = [phase, cont, onehot]
    blocks += [base.rolling(w, min_periods=1).mean().to_numpy() for w in (6,24,72)]
    blocks += [base.diff(24).fillna(0).to_numpy()]
    m = np.concatenate(blocks, 1)
    m = m[:, m.std(0) > 1e-8]
    mu, sd = m[:fit_end].mean(0), m[:fit_end].std(0) + 1e-8
    return ((m - mu)/sd).astype(np.float32)

class Scaler:
    def __init__(self, y_fit): self.mu, self.sd = float(y_fit.mean()), float(y_fit.std())
    def transform(self, y): return (y - self.mu)/self.sd
    def inverse(self, v):   return v*self.sd + self.mu

cpu 43656 43824 val from 34920


In [3]:
class SeriesDecomposition(nn.Module):
    """Centered moving average, replicated ends (from Question 1)."""
    def __init__(self, kernel): super().__init__(); self.kernel = kernel
    def forward(self, x):
        pad = self.kernel//2
        p = F.pad(x.transpose(1,2), (pad,pad), mode="replicate")
        trend = F.avg_pool1d(p, self.kernel, stride=1).transpose(1,2)
        return x - trend, trend

def delay_scores(q, k):
    q = q - q.mean(-1, keepdim=True); k = k - k.mean(-1, keepdim=True)
    s = torch.fft.rfft(q, dim=-1) * torch.fft.rfft(k, dim=-1).conj()
    return torch.fft.irfft(s, n=q.shape[-1], dim=-1).mean((1,2))

def aggregate_delays(v, delays, weights):
    B,h,d,L = v.shape
    pos = torch.arange(L, device=v.device); out = torch.zeros_like(v)
    for j in range(delays.shape[1]):
        src = ((pos[None,:] - delays[:, j:j+1]) % L)[:,None,None,:].expand(B,h,d,L)
        out = out + weights[:, j,None,None,None] * v.gather(-1, src)
    return out

In [4]:
class AutoCorrelation(nn.Module):
    def __init__(self, d, heads, factor):
        super().__init__(); self.heads, self.factor = heads, factor
        self.q, self.k, self.v, self.out = (nn.Linear(d,d) for _ in range(4))
        self.raw_temp = nn.Parameter(torch.tensor(0.5413))
    def forward(self, qx, kx, vx):
        B,L,d = qx.shape; S = kx.shape[1]
        if S < L:
            pad = torch.zeros(B, L-S, d, device=kx.device, dtype=kx.dtype)
            kx, vx = torch.cat([kx,pad],1), torch.cat([vx,pad],1)
        elif S > L: kx, vx = kx[:,:L], vx[:,:L]
        sh = (B,L,self.heads,d//self.heads)
        q = self.q(qx).view(sh).permute(0,2,3,1); k = self.k(kx).view(sh).permute(0,2,3,1)
        v = self.v(vx).view(sh).permute(0,2,3,1)
        sc = delay_scores(q,k)/L
        sel, dl = sc.topk(max(1,int(self.factor*math.log(L))), -1)
        w = (sel/(F.softplus(self.raw_temp)+1e-6)).softmax(-1)
        m = aggregate_delays(v, dl, w)
        return self.out(m.permute(0,3,1,2).reshape(B,L,d))

class SeasonalNorm(nn.Module):
    def __init__(self, d): super().__init__(); self.n = nn.LayerNorm(d)
    def forward(self, x): x = self.n(x); return x - x.mean(1, keepdim=True)

class Embed(nn.Module):
    def __init__(self, n_marks, d, p):
        super().__init__()
        self.val = nn.Conv1d(1, d, 3, padding=1, padding_mode="circular", bias=False)
        self.mk = nn.Linear(n_marks, d, bias=False); self.drop = nn.Dropout(p)
    def forward(self, x, m):
        return self.drop(self.val(x.transpose(1,2)).transpose(1,2) + self.mk(m))

class EncLayer(nn.Module):
    def __init__(self, d,h,dff,k,f,p):
        super().__init__(); self.att = AutoCorrelation(d,h,f)
        self.ff = nn.Sequential(nn.Linear(d,dff,bias=False), nn.GELU(), nn.Dropout(p),
                                nn.Linear(dff,d,bias=False))
        self.d1, self.d2 = SeriesDecomposition(k), SeriesDecomposition(k); self.drop = nn.Dropout(p)
    def forward(self, x):
        x,_ = self.d1(x + self.drop(self.att(x,x,x)))
        x,_ = self.d2(x + self.drop(self.ff(x))); return x

class DecLayer(nn.Module):
    def __init__(self, d,h,dff,k,f,p):
        super().__init__()
        self.sa, self.ca = AutoCorrelation(d,h,f), AutoCorrelation(d,h,f)
        self.ff = nn.Sequential(nn.Linear(d,dff,bias=False), nn.GELU(), nn.Dropout(p),
                                nn.Linear(dff,d,bias=False))
        self.d1,self.d2,self.d3 = (SeriesDecomposition(k) for _ in range(3))
        self.tp = nn.Conv1d(d,1,3,padding=1,padding_mode="circular",bias=False); self.drop = nn.Dropout(p)
    def forward(self, x, mem):
        x,t1 = self.d1(x + self.drop(self.sa(x,x,x)))
        x,t2 = self.d2(x + self.drop(self.ca(x,mem,mem)))
        x,t3 = self.d3(x + self.drop(self.ff(x)))
        return x, self.tp((t1+t2+t3).transpose(1,2)).transpose(1,2)

class Autoformer(nn.Module):
    """Autoformer (Wu et al. 2021) using the Question 1 decomposition and delay components.
    cov_hidden>0 adds a pointwise path for the horizon covariates, which the circular
    delay aggregation would otherwise scramble."""
    def __init__(self, n_marks, d_model=32, n_heads=4, e_layers=1, d_layers=1, d_ff=64,
                 kernel=25, factor=2, dropout=0.05, cov_hidden=32):
        super().__init__()
        self.decomp = SeriesDecomposition(kernel)
        self.ee, self.de = Embed(n_marks,d_model,dropout), Embed(n_marks,d_model,dropout)
        self.enc = nn.ModuleList([EncLayer(d_model,n_heads,d_ff,kernel,factor,dropout) for _ in range(e_layers)])
        self.en = SeasonalNorm(d_model)
        self.dec = nn.ModuleList([DecLayer(d_model,n_heads,d_ff,kernel,factor,dropout) for _ in range(d_layers)])
        self.dn = SeasonalNorm(d_model); self.head = nn.Linear(d_model,1)
        self.cov = nn.Sequential(nn.Linear(n_marks,cov_hidden), nn.GELU(),
                                 nn.Linear(cov_hidden,1)) if cov_hidden else None
    def forward(self, x_enc, m_enc, m_dec):
        B = x_enc.shape[0]
        seasonal, trend = self.decomp(x_enc)
        mean = x_enc.mean(1, keepdim=True).expand(B, PRED_LEN, 1)
        trend = torch.cat([trend[:, -LABEL_LEN:], mean], 1)
        seasonal = torch.cat([seasonal[:, -LABEL_LEN:], torch.zeros_like(mean)], 1)
        mem = self.ee(x_enc, m_enc)
        for l in self.enc: mem = l(mem)
        mem = self.en(mem)
        x = self.de(seasonal, m_dec)
        for l in self.dec:
            x, rt = l(x, mem); trend = trend + rt
        out = trend + self.head(self.dn(x))
        if self.cov is not None: out = out + self.cov(m_dec)
        return out[:, -PRED_LEN:, 0]

count_parameters = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)

In [5]:
class Data:
    def __init__(self, fit_end, use_external=True, horizon_cov=True):
        marks = build_marks(fit_end, use_external)
        self.n_marks = marks.shape[1]; self.horizon_cov = horizon_cov
        self.scaler = Scaler(y_all[:fit_end])
        self.z = torch.tensor(self.scaler.transform(y_all), dtype=torch.float32, device=DEVICE)
        self.m = torch.tensor(marks, device=DEVICE)
        self.ae = torch.arange(-SEQ_LEN,0,device=DEVICE)
        self.ad = torch.arange(-LABEL_LEN,PRED_LEN,device=DEVICE)
        self.at = torch.arange(0,PRED_LEN,device=DEVICE)
    def inputs(self, o):
        o = o[:,None]; md = self.m[o+self.ad]
        if not self.horizon_cov: md = md.clone(); md[:, LABEL_LEN:] = 0.0
        return self.z[o+self.ae].unsqueeze(-1), self.m[o+self.ae], md
    def targets(self, o): return self.z[o[:,None]+self.at]

@torch.no_grad()
def predict(model, data, origins):
    model.eval(); o = torch.as_tensor(origins, device=DEVICE)
    return np.concatenate([data.scaler.inverse(model(*data.inputs(o[i:i+64]))).cpu().numpy()
                           for i in range(0,len(o),64)])

def metrics(p, truth=TRUE_VAL):
    p = np.clip(p, 0, None); e = p - truth
    return dict(RMSE=float(np.sqrt((e**2).mean())), MAE=float(np.abs(e).mean()),
                sMAPE=float(100*np.mean(2*np.abs(e)/(np.abs(truth)+np.abs(p)+1e-8))))

STRIDE = 4   # training set = every 4th window; neighbouring windows overlap 99%
def train(data, train_end, epochs, seed, lr=3e-4, batch=64, model_kwargs=None,
          eval_origins=None, log=False):
    torch.manual_seed(seed); np.random.seed(seed)
    model = Autoformer(data.n_marks, **(model_kwargs or {})).to(DEVICE)
    origins = torch.arange(SEQ_LEN, train_end-PRED_LEN+1, STRIDE, device=DEVICE)
    steps = epochs*math.ceil(len(origins)/batch)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.15)
    gen = torch.Generator().manual_seed(seed); hist = []
    for ep in range(1, epochs+1):
        model.train(); t0 = time.time()
        perm = origins[torch.randperm(len(origins), generator=gen).to(DEVICE)]
        for i in range(0, len(perm), batch):
            o = perm[i:i+batch]
            loss = F.mse_loss(model(*data.inputs(o)), data.targets(o))
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sch.step()
        row = {"epoch": ep, "sec": round(time.time()-t0,1)}
        if eval_origins is not None: row.update({k: round(v,2) for k,v in
                                                 metrics(predict(model,data,eval_origins)).items()})
        hist.append(row)
        if log: print("   ", row, flush=True)
    return model, hist

In [6]:
print("training-mean baseline RMSE:", round(metrics(np.full_like(TRUE_VAL, y_all[:VAL_START].mean()))["RMSE"],2))
d = Data(VAL_START)
m, h = train(d, VAL_START, 8, 0, eval_origins=VAL_ORIGINS, log=True)
print("parameters:", count_parameters(m))

training-mean baseline RMSE: 92.98
    {'epoch': 1, 'sec': 37.8, 'RMSE': 79.19, 'MAE': 56.2, 'sMAPE': 69.82}
    {'epoch': 2, 'sec': 37.0, 'RMSE': 73.38, 'MAE': 52.43, 'sMAPE': 66.67}
    {'epoch': 3, 'sec': 36.6, 'RMSE': 69.91, 'MAE': 50.23, 'sMAPE': 67.34}
    {'epoch': 4, 'sec': 37.1, 'RMSE': 69.1, 'MAE': 49.83, 'sMAPE': 68.45}
    {'epoch': 5, 'sec': 38.2, 'RMSE': 67.97, 'MAE': 48.9, 'sMAPE': 68.42}
    {'epoch': 6, 'sec': 38.4, 'RMSE': 67.13, 'MAE': 48.16, 'sMAPE': 67.71}
    {'epoch': 7, 'sec': 38.1, 'RMSE': 67.16, 'MAE': 48.61, 'sMAPE': 67.9}
    {'epoch': 8, 'sec': 43.2, 'RMSE': 66.98, 'MAE': 48.44, 'sMAPE': 67.84}
parameters: 26565


In [7]:
BEST_EPOCHS = 5     # <-- set from Cell 6
SEEDS = (0,1,2)
for label, kw in {"horizon covariates": dict(use_external=True, horizon_cov=True),
                  "history only":       dict(use_external=True, horizon_cov=False),
                  "no external":        dict(use_external=False)}.items():
    rows = []
    for s in SEEDS:
        d = Data(VAL_START, **kw)
        m,h = train(d, VAL_START, BEST_EPOCHS, s, eval_origins=VAL_ORIGINS)
        rows.append(h[-1])
    r = [x["RMSE"] for x in rows]
    print(f"{label:20s} RMSE {np.mean(r):6.2f} (sd {np.std(r):.2f}) "
          f"MAE {np.mean([x['MAE'] for x in rows]):6.2f} "
          f"sMAPE {np.mean([x['sMAPE'] for x in rows]):5.2f} | params {count_parameters(m)}")

horizon covariates   RMSE  69.41 (sd 0.88) MAE  50.15 sMAPE 67.24 | params 26565
history only         RMSE  95.85 (sd 0.92) MAE  70.67 sMAPE 75.54 | params 26565
no external          RMSE  96.13 (sd 0.52) MAE  70.60 sMAPE 74.99 | params 21765


In [8]:
val_preds, final_preds = [], []
for s in SEEDS:
    dv = Data(VAL_START); mv,_ = train(dv, VAL_START, BEST_EPOCHS, s)
    val_preds.append(predict(mv, dv, VAL_ORIGINS))
    df = Data(N); mf,_ = train(df, N, BEST_EPOCHS, s)
    final_preds.append(predict(mf, df, [N])[0])
    print("seed", s, "done", flush=True)

P1, E1 = count_parameters(mf), BEST_EPOCHS
single = [metrics(p)["RMSE"] for p in val_preds]
ens = metrics(np.mean(val_preds,0))["RMSE"]
print(f"\nsingle seed: mean {np.mean(single):.2f} sd {np.std(single):.2f}")
print(f"{len(SEEDS)}-seed ensemble: {ens:.2f}")
print(f"single  -> P={P1} E={E1}")
print(f"ensemble-> P={P1*len(SEEDS)} E={E1*len(SEEDS)}")

USE_ENSEMBLE = True      # <-- set False if the ensemble gain is under ~2 RMSE
pred = np.clip(np.mean(final_preds,0) if USE_ENSEMBLE else final_preds[0], 6, None)
print(f"\n168 values | min {pred.min():.1f} max {pred.max():.1f} mean {pred.mean():.1f}")
print("DECLARE P =", P1*len(SEEDS) if USE_ENSEMBLE else P1,
      " E =", E1*len(SEEDS) if USE_ENSEMBLE else E1)
print("\n" + ",".join(f"{v:.3f}" for v in pred))

seed 0 done
seed 1 done
seed 2 done

single seed: mean 69.41 sd 0.88
3-seed ensemble: 67.24
single  -> P=26565 E=5
ensemble-> P=79695 E=15

168 values | min 6.0 max 239.6 mean 109.7
DECLARE P = 79695  E = 15

6.328,6.000,6.000,6.000,6.000,6.000,6.000,6.000,6.000,6.000,8.856,9.540,7.948,6.420,12.223,14.003,14.820,31.502,44.141,60.476,64.841,77.029,81.232,82.688,80.525,88.361,91.980,92.161,105.256,103.315,100.179,91.839,86.721,106.230,109.400,113.933,117.508,129.907,138.315,144.009,163.483,176.922,195.114,196.455,201.699,211.328,223.472,217.073,214.761,202.489,191.289,170.160,158.723,145.779,142.536,134.251,130.986,118.703,116.374,103.789,98.235,121.735,129.408,134.019,142.746,149.758,177.330,196.732,194.097,216.703,231.898,235.563,229.667,239.618,233.753,214.313,195.676,175.474,153.608,139.356,149.815,149.170,143.932,136.069,137.423,132.412,117.454,106.091,102.225,100.970,99.596,92.490,128.486,139.549,155.783,181.387,193.124,187.765,175.411,169.893,163.998,161.172,169.993,163.568,162.64

In [9]:
ens = np.mean(val_preds, 0)
for f in (0, 3, 6, 10, 15, 20, 25):
    print(f"floor {f:3d}: ensemble RMSE {metrics(np.clip(ens, f, None))['RMSE']:.2f} "
          f"| single-seed mean {np.mean([metrics(np.clip(p, f, None))['RMSE'] for p in val_preds]):.2f}")
print("\nunclamped final forecast: min", round(np.mean(final_preds,0).min(),1),
      "| values below 15:", int((np.mean(final_preds,0) < 15).sum()))

floor   0: ensemble RMSE 67.24 | single-seed mean 69.41
floor   3: ensemble RMSE 67.20 | single-seed mean 69.35
floor   6: ensemble RMSE 67.16 | single-seed mean 69.30
floor  10: ensemble RMSE 67.11 | single-seed mean 69.23
floor  15: ensemble RMSE 67.06 | single-seed mean 69.15
floor  20: ensemble RMSE 67.03 | single-seed mean 69.10
floor  25: ensemble RMSE 67.03 | single-seed mean 69.08

unclamped final forecast: min -61.7 | values below 15: 39


In [11]:
FLOOR = 10             # <-- set from Cell 9
USE_ENSEMBLE = True

pred = np.clip(np.mean(final_preds, 0) if USE_ENSEMBLE else final_preds[0], FLOOR, None)
print(f"168 values | min {pred.min():.1f} max {pred.max():.1f} mean {pred.mean():.1f}")
print("DECLARE P =", P1*len(SEEDS) if USE_ENSEMBLE else P1,
      " E =", E1*len(SEEDS) if USE_ENSEMBLE else E1)
print("\n" + ",".join(f"{v:.3f}" for v in pred))

168 values | min 10.0 max 239.6 mean 110.5
DECLARE P = 79695  E = 15

10.000,10.000,10.000,10.000,10.000,10.000,10.000,10.000,10.000,10.000,10.000,10.000,10.000,10.000,12.223,14.003,14.820,31.502,44.141,60.476,64.841,77.029,81.232,82.688,80.525,88.361,91.980,92.161,105.256,103.315,100.179,91.839,86.721,106.230,109.400,113.933,117.508,129.907,138.315,144.009,163.483,176.922,195.114,196.455,201.699,211.328,223.472,217.073,214.761,202.489,191.289,170.160,158.723,145.779,142.536,134.251,130.986,118.703,116.374,103.789,98.235,121.735,129.408,134.019,142.746,149.758,177.330,196.732,194.097,216.703,231.898,235.563,229.667,239.618,233.753,214.313,195.676,175.474,153.608,139.356,149.815,149.170,143.932,136.069,137.423,132.412,117.454,106.091,102.225,100.970,99.596,92.490,128.486,139.549,155.783,181.387,193.124,187.765,175.411,169.893,163.998,161.172,169.993,163.568,162.640,164.017,163.230,160.154,160.846,144.439,127.120,116.457,103.476,95.716,127.845,144.489,179.330,187.908,197.154,221.785,214.